# Task 01 — Text Generation with GPT-2 (Fine-Tuning





In [ ]:
!pip uninstall -y transformers
!pip install transformers==4.46.3 accelerate==1.1.1 datasets==3.1.0 evaluate==0.4.3

Found existing installation: transformers 4.46.3
Uninstalling transformers-4.46.3:
  Successfully uninstalled transformers-4.46.3
  Using cached transformers-4.46.3-py3-none-any.whl.metadata (44 kB)
Using cached transformers-4.46.3-py3-none-any.whl (10.0 MB)


In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


## 1. Install dependencies

In [ ]:
!pip install -q transformers datasets accelerate evaluate


In [ ]:
import torch
import random
import numpy as np
import math

from datasets import load_dataset
from transformers import (
    GPT2LMHeadModel,
    GPT2Tokenizer,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)


The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

Using device: cuda


## 2. Load the dataset




In [ ]:
# Use a manageable subset for a fast fine-tune on a free Colab GPU.
# Increase these numbers if you have more time/compute.
TRAIN_SIZE = 4000
VAL_SIZE = 400

raw_dataset = load_dataset("roneneldan/TinyStories")

train_data = raw_dataset["train"].shuffle(seed=SEED).select(range(TRAIN_SIZE))
val_data = raw_dataset["validation"].shuffle(seed=SEED).select(range(VAL_SIZE))

print(train_data)
print(val_data)
print("\nSample story:\n", train_data[0]["text"])


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Dataset({
    features: ['text'],
    num_rows: 4000
})
Dataset({
    features: ['text'],
    num_rows: 400
})

Sample story:
 Tim and Mia like to play in the park. They see a big club on the ground. It is brown and long and heavy.

"Look, a club!" Tim says. "I can lift it!"

He tries to lift the club, but it is too tough. He falls down and drops the club.

"Ouch!" he says. "That hurt!"

Mia laughs. She is not mean, she just thinks it is funny.

"Let me try!" she says. "I can balance it!"

She picks up the club and puts it on her head. She walks slowly and carefully. She does not fall down.

"Wow!" Tim says. "You are good at balancing!"

"Thank you!" Mia says. "It is fun!"

They take turns balancing the club on their heads, arms, and legs. They have a lot of fun with the club. They are happy and proud. They are good friends.


## 3. Decoding strategies on base GPT-2




In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

base_tokenizer = AutoTokenizer.from_pretrained("gpt2")
base_model = AutoModelForCausalLM.from_pretrained(
    "gpt2", pad_token_id=base_tokenizer.eos_token_id
).to(device)

demo_prompt = "The old lighthouse keeper walked down to the shore and"
demo_inputs = base_tokenizer(demo_prompt, return_tensors="pt").to(device)

def show(label, output_ids):
    print(f"--- {label} ---")
    print(base_tokenizer.decode(output_ids[0], skip_special_tokens=True))
    print()

# 1. Greedy search
out = base_model.generate(**demo_inputs, max_new_tokens=50)
show("Greedy search", out)

# 2. Beam search with a repeat n-gram penalty
out = base_model.generate(**demo_inputs, max_new_tokens=50, num_beams=5,
                           no_repeat_ngram_size=2, early_stopping=True)
show("Beam search (5 beams, no_repeat_ngram_size=2)", out)

# 3. Pure sampling (no top-k / top-p restriction)
set_seed(42)
out = base_model.generate(**demo_inputs, max_new_tokens=50, do_sample=True, top_k=0)
show("Pure sampling", out)

# 4. Sampling with temperature
set_seed(42)
out = base_model.generate(**demo_inputs, max_new_tokens=50, do_sample=True, top_k=0, temperature=0.6)
show("Sampling, temperature=0.6", out)

# 5. Top-k sampling
set_seed(42)
out = base_model.generate(**demo_inputs, max_new_tokens=50, do_sample=True, top_k=50)
show("Top-k sampling (k=50)", out)

# 6. Top-p (nucleus) sampling
set_seed(42)
out = base_model.generate(**demo_inputs, max_new_tokens=50, do_sample=True, top_k=0, top_p=0.92)
show("Top-p sampling (p=0.92)", out)


Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


--- Greedy search ---
The old lighthouse keeper walked down to the shore and looked at the old lighthouse keeper.

"I'm sorry, I'm sorry," he said. "I'm sorry. I'm sorry. I'm sorry. I'm sorry. I'm sorry. I'm sorry. I'm sorry.



Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


--- Beam search (5 beams, no_repeat_ngram_size=2) ---
The old lighthouse keeper walked down to the shore and looked up at the sky.

"What's going on here?" he asked. "I don't know what's happening, but it looks like we're in the middle of something big. I can't tell you what it is."




Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


--- Pure sampling ---
The old lighthouse keeper walked down to the shore and forth to take a picture as the Chief wished to have taken some pictures of it.

CONNECTION 6

One of the strangest pictures in History is a solitary old woman standing in the *clubway* whilst holding a clock face



Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


--- Sampling, temperature=0.6 ---
The old lighthouse keeper walked down to the shore and saw the old man standing in the doorway. "It's hard to see," she said, "but I can tell you that the old man had been there." She said she'd seen the old man, but he'd never been there before.



Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


--- Top-k sampling (k=50) ---
The old lighthouse keeper walked down to the shore and saw the old man who had just turned around had been taken in by another man she could not locate with the help of the mirror. She asked him for a few moments and then said how long it has been since he has returned to the lighthouse.

--- Top-p sampling (p=0.92) ---
The old lighthouse keeper walked down to the shore and forth to take a picture. "This isn't the first time I've seen this island, let's not kid ourselves!" A hawk looked at him for a few moments, then immediately walked back toward him. "Now, Mr. Andersen! There



## 4. Tokenize and chunk the text

In [ ]:
MODEL_NAME = "gpt2"          # swap for "gpt2-medium" if you have a bigger GPU
BLOCK_SIZE = 256              # sequence length per training example

tokenizer = GPT2Tokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token   # GPT-2 has no pad token by default

def tokenize_fn(examples):
    return tokenizer(examples["text"])

tokenized_train = train_data.map(tokenize_fn, batched=True, remove_columns=train_data.column_names)
tokenized_val = val_data.map(tokenize_fn, batched=True, remove_columns=val_data.column_names)

def group_texts(examples):
    # Concatenate all texts
    concatenated_examples = {}
    for k in examples.keys():
        concatenated_examples[k] = sum(examples[k], [])

    # Make all columns the same length
    total_length = min(len(v) for v in concatenated_examples.values())

    # Drop the remainder
    total_length = (total_length // BLOCK_SIZE) * BLOCK_SIZE

    # Split into chunks
    result = {
        k: [v[i:i + BLOCK_SIZE] for i in range(0, total_length, BLOCK_SIZE)]
        for k, v in concatenated_examples.items()
    }

    result["labels"] = result["input_ids"].copy()

    return result

lm_train = tokenized_train.map(group_texts, batched=True)
lm_val = tokenized_val.map(group_texts, batched=True)

print(f"Training chunks: {len(lm_train)}  |  Validation chunks: {len(lm_val)}")


Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (1087 > 1024). Running this sequence through the model will result in indexing errors


Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Training chunks: 3487  |  Validation chunks: 339


## 5. Fine-tune GPT-2

In [ ]:
model = GPT2LMHeadModel.from_pretrained(MODEL_NAME).to(device)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir="./gpt2-shortstories",
    overwrite_output_dir=True,
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    logging_steps=50,
    learning_rate=5e-5,
    warmup_steps=100,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=lm_train,
    eval_dataset=lm_val,
    data_collator=data_collator,
)

trainer.train()


Epoch,Training Loss,Validation Loss
1,2.005900,1.925363
2,1.905400,1.868157
3,1.834300,1.855391


TrainOutput(global_step=1308, training_loss=1.9641732520648827, metrics={'train_runtime': 413.7563, 'train_samples_per_second': 25.283, 'train_steps_per_second': 3.161, 'total_flos': 1366687973376000.0, 'train_loss': 1.9641732520648827, 'epoch': 3.0})

In [ ]:
import transformers
print(transformers.__version__)

4.46.3


## 6. Save the fine-tuned model

In [ ]:
SAVE_DIR = "./gpt2-shortstories-final"
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print("Model saved to", SAVE_DIR)


Model saved to ./gpt2-shortstories-final


## 7. Generate text from prompts


In [ ]:
gen_tokenizer = GPT2Tokenizer.from_pretrained(SAVE_DIR)
gen_model = GPT2LMHeadModel.from_pretrained(SAVE_DIR).to(device)
gen_model.eval()

def generate_text(prompt, max_new_tokens=120, temperature=0.8, top_k=50, top_p=0.95, num_return_sequences=1):
    input_ids = gen_tokenizer.encode(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        output = gen_model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_k=top_k,
            top_p=top_p,
            num_return_sequences=num_return_sequences,
            pad_token_id=gen_tokenizer.eos_token_id,
        )
    return [gen_tokenizer.decode(seq, skip_special_tokens=True) for seq in output]

prompts = [
    "Once upon a time, in a small village,",
    "The little robot looked at the stars and",
    "Every morning, the old cat would",
]

for p in prompts:
    print("PROMPT:", p)
    for out in generate_text(p):
        print("→", out)
    print("-" * 80)


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


PROMPT: Once upon a time, in a small village,
→ Once upon a time, in a small village, there lived a boy named Sam. He loved to play with his toys, and he always had a big toy on hand. One day, Sam decided to go to the park with his mom. 

Sam walked down the path, and when he came to the playground, the boy saw a big slide. He quickly hopped up and down, and he jumped right in. 

After a few more swings, Sam jumped in front of the slide. He smiled and felt so proud of himself for being able to do it. 

As Sam watched, he said, "Mom
--------------------------------------------------------------------------------
PROMPT: The little robot looked at the stars and
→ The little robot looked at the stars and said, "Look, there's a bright star in the sky!" The little robot was excited and took the star. It showed the little robot to everyone in the garden. The little robot was so happy that the little robot was not scared anymore.Once upon a time, there was a little girl named Lily. She loved 

## 8. Decoding strategy comparison — Greedy vs. Beam Search vs. Sampling


In [ ]:
from transformers import set_seed

base_tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
base_model = GPT2LMHeadModel.from_pretrained("gpt2", pad_token_id=base_tokenizer.eos_token_id).to(device)
base_model.eval()

def run_decoding_strategies(model, tokenizer, prompt, max_new_tokens=60, seed=42):
    model_inputs = tokenizer(prompt, return_tensors="pt").to(device)
    results = {}

    # 1. Greedy search
    set_seed(seed)
    out = model.generate(**model_inputs, max_new_tokens=max_new_tokens)
    results["Greedy search"] = tokenizer.decode(out[0], skip_special_tokens=True)

    # 2. Beam search with no-repeat n-gram penalty
    set_seed(seed)
    out = model.generate(
        **model_inputs, max_new_tokens=max_new_tokens,
        num_beams=5, no_repeat_ngram_size=2, early_stopping=True,
    )
    results["Beam search (num_beams=5, no_repeat_ngram_size=2)"] = tokenizer.decode(out[0], skip_special_tokens=True)

    # 3. Sampling with temperature
    set_seed(seed)
    out = model.generate(
        **model_inputs, max_new_tokens=max_new_tokens,
        do_sample=True, top_k=0, temperature=0.7,
    )
    results["Sampling (temperature=0.7)"] = tokenizer.decode(out[0], skip_special_tokens=True)

    # 4. Top-K sampling
    set_seed(seed)
    out = model.generate(
        **model_inputs, max_new_tokens=max_new_tokens,
        do_sample=True, top_k=50,
    )
    results["Top-K sampling (top_k=50)"] = tokenizer.decode(out[0], skip_special_tokens=True)

    # 5. Top-p (nucleus) sampling
    set_seed(seed)
    out = model.generate(
        **model_inputs, max_new_tokens=max_new_tokens,
        do_sample=True, top_k=0, top_p=0.92,
    )
    results["Top-p sampling (top_p=0.92)"] = tokenizer.decode(out[0], skip_special_tokens=True)

    return results

comparison_prompt = "Once upon a time, in a small village,"

print("=" * 100)
print("BASE GPT-2 (before fine-tuning)")
print("=" * 100)
base_results = run_decoding_strategies(base_model, base_tokenizer, comparison_prompt)
for strategy, text in base_results.items():
    print(f"\n[{strategy}]\n{text}")

print("\n\n" + "=" * 100)
print("FINE-TUNED GPT-2 (after fine-tuning on TinyStories)")
print("=" * 100)
finetuned_results = run_decoding_strategies(gen_model, gen_tokenizer, comparison_prompt)
for strategy, text in finetuned_results.items():
    print(f"\n[{strategy}]\n{text}")


BASE GPT-2 (before fine-tuning)


Setting `pad_token_id` to `eos_token_id`:None for open-end generation.



[Greedy search]
Once upon a time, in a small village, a young man named Tzal, who had been a member of the village's council, was killed by a group of bandits. He was taken to the village of Tzal, where he was taken to the village of Tzal's father, who was killed by the bandits.


[Beam search (num_beams=5, no_repeat_ngram_size=2)]
Once upon a time, in a small village, there was a man who had been killed by a group of bandits.

The man's body was found in the forest. The bandits had killed the man, but he was still alive. He was the son of a nobleman, and the father of one of the noblemen who was killed

[Sampling (temperature=0.7)]
Once upon a time, in a small village, for the first time in history, a fire had been created in a village called Mokushu, the home of the Aksumin. The Mokushu clan had been led by the Ryuku clan, and it was here where they came to rule the world. In the village

[Top-K sampling (top_k=50)]
Once upon a time, in a small village, someone was playing cards in 

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.



[Greedy search]
Once upon a time, in a small village, there lived a little girl named Lily. She loved to play outside and explore the world around her. One day, she saw a big, scary monster. She wanted to help it, so she ran to the monster and tried to stop it. But the monster was too fast and it was too fast

[Beam search (num_beams=5, no_repeat_ngram_size=2)]
Once upon a time, in a small village, there lived a little girl named Lily. She loved to play with her toys and have fun. One day, Lily's mommy asked her if she wanted to go on an adventure. Lily said yes, and they went to the park. 

When they got there, they saw a big

[Sampling (temperature=0.7)]
Once upon a time, in a small village, there was a girl named Lily. She loved to play outside in the grass. One day, she found a big tree with many leaves and branches. She picked up the leaves and put them in a big bowl.

Suddenly, a big gust of wind came and blew the leaves away. Lily

[Top-K sampling (top_k=50)]
Once upon a time, 

**What to look for:** the fine-tuned model's outputs should read noticeably more like a short
story (simple narrative sentences, story-like continuations) compared to the base model, across every
decoding strategy — while greedy/beam search should still show more repetition than the sampling-based
methods, consistent with the Hugging Face guide.


## 9. Evaluate with perplexity

In [ ]:
def compute_perplexity(trainer, eval_dataset):
    metrics = trainer.evaluate(eval_dataset=eval_dataset)
    eval_loss = metrics["eval_loss"]
    perplexity = math.exp(eval_loss)
    print(f"Eval loss: {eval_loss:.4f}  |  Perplexity: {perplexity:.2f}")
    return perplexity

perplexity = compute_perplexity(trainer, lm_val)


Eval loss: 1.8554  |  Perplexity: 6.39


## 10. Try your own prompt


In [ ]:
your_prompt = "The dragon flew over the mountains and"  #@param {type:"string"}
for out in generate_text(your_prompt, num_return_sequences=3):
    print("→", out)
    print("-" * 80)


→ The dragon flew over the mountains and saw them. It was a bright red dragon and it was flying high in the sky. The dragon was so surprised. It was so excited that it flew up high in the sky.

The dragon flew high in the sky. It made a big rainbow around it. It was so happy and excited. It was so happy that it flew up high in the sky. It flew so fast that it made a rainbow across the sky. It was so happy that it flew up high in the sky.

The dragon flew up high in the sky again. It flew so fast that it made
--------------------------------------------------------------------------------
→ The dragon flew over the mountains and landed in a clearing. It was so beautiful! It looked up at the sky and said, â€œLook, I'm here!â€

The dragon flew around the forest and then flew back to its home. It took a nap in a clearing and then rested in its cozy nest. It felt happy and full.Once upon a time, there was a brave little girl named Lily. She loved to play outside in the wild. One day, she sa

## 11. Decoding strategies on the fine-tuned model


In [ ]:
demo_inputs_ft = gen_tokenizer(demo_prompt, return_tensors="pt").to(device)

def show_ft(label, output_ids):
    print(f"--- {label} ---")
    print(gen_tokenizer.decode(output_ids[0], skip_special_tokens=True))
    print()

out = gen_model.generate(**demo_inputs_ft, max_new_tokens=50, pad_token_id=gen_tokenizer.eos_token_id)
show_ft("Greedy search (fine-tuned)", out)

out = gen_model.generate(**demo_inputs_ft, max_new_tokens=50, num_beams=5, no_repeat_ngram_size=2,
                          early_stopping=True, pad_token_id=gen_tokenizer.eos_token_id)
show_ft("Beam search (fine-tuned)", out)

set_seed(42)
out = gen_model.generate(**demo_inputs_ft, max_new_tokens=50, do_sample=True, top_k=50, top_p=0.95,
                          pad_token_id=gen_tokenizer.eos_token_id)
show_ft("Top-k + Top-p sampling (fine-tuned)", out)


--- Greedy search (fine-tuned) ---
The old lighthouse keeper walked down to the shore and saw a big, red boat. He was so excited to see the boat. He said, "Look, the boat is so big!"

The old lighthouse keeper smiled and said, "That's a great idea, but I have to ask

--- Beam search (fine-tuned) ---
The old lighthouse keeper walked down to the shore and saw a little boy. The boy was very curious and wanted to know more about the lighthouse.

"Hello, boy," said the old keeper. "Do you want to go on a tour of the island?" 
The boy nodded and

--- Top-k + Top-p sampling (fine-tuned) ---
The old lighthouse keeper walked down to the shore and saw the little girl who lived in the park. He saw her walking towards it and felt jealous.

"Can you be my keeper?" asked the little girl, waving goodbye.

"I can be you too, Sam. I can

